In [ ]:
import random
from copy import deepcopy
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
from sklearn.metrics import r2_score
from torch_geometric.data import Data
from tqdm import tqdm

from barostat_utils import (
    estimate_initial_box_vel_y_accurate,
    update_box_y_thermodynamic,
)
from graph_utils import prepare_traj
from itpo_weights import DatasetType
from pressure import compute_virial_stress
from utils import (
    calc_p_ratio_box_tensor,
    load_and_split_dataset,
)

### Data

In [ ]:
poisson_buckets = [
    {"max": 0.1, "count": 100},                # P < 0.1
    {"min": 0.1, "max": 0.2, "count": 100},    # 0.1 <= P < 0.2
    {"min": 0.2, "count": 200}                 # P >= 0.2
]

dataset_type = DatasetType.NodeOptimized

train_files, val_files, test_files = load_and_split_dataset(
    registry_path="./data_mini/data_registry_mini.csv",
    target_data_type=dataset_type,
    possion_buckets=poisson_buckets,
    split_ratios=(0.5, 0.25, 0.25),
    seed=42
)

# Load actual data
data = {
    'train': {},
    'val' : {},
    'test' : {},
}

print("Loading data...")
for key in data.keys():
    if key == 'train':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(train_files, desc=f"{key:<5} data")]
    elif key == 'val':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(val_files, desc=f"{key:<5} data")]
    elif key == 'test':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(test_files, desc=f"{key:<5} data")]
    else:
        raise ValueError(f"Unexpected key in data dictionary: {key}. ")

print("\nPreparing data...")
for data_type, sims in data.items():
    prepared = []
    for sim in tqdm(sims, desc=f"{data_type:<5} data"):
        prepared_sim = prepare_traj(sim, calc_angles=False)
        prepared.append(prepared_sim)
    data[data_type] = prepared

print(f"\nTrain data: {len(data['train'])} sims.")
print(f"Val data:   {len(data['val'])} sims.")
print(f"Test data:  {len(data['test'])} sims.")


#### Show $\nu$ distribution

In [ ]:
def visualize_nu_disribution(data: Dict):
    ps = {}
    all_values = []

    for data_type, sims in data.items():
        ps[data_type] = [calc_p_ratio_box_tensor(sim).item() for sim in sims]
        all_values.extend(ps[data_type])

    min_val = min(all_values)
    max_val = max(all_values)
    common_bins = np.linspace(min_val, max_val, 30) 

    for data_type, values in ps.items():
        plt.hist(
        values, 
        bins=common_bins, 
        edgecolor='black', 
        alpha=0.6, 
        label=f"{data_type} data"
    )

    plt.title("$\\nu$ distribution")
    plt.xlabel("GT LAMMPS $\\nu$")
    plt.ylabel("N")
    plt.legend()
    plt.show()

visualize_nu_disribution(data)


### Barostat parameter tuning

In [ ]:
def run_trajectory_rollout(
    sim: List[Data],
    c_coupling: float,
    damping_coeff: float,
    stride_dt: float,
    temperature: float = 1e-7
):
    """
    Simulates Box evolution over N steps using GT positions.
    """
    sim = [g.cpu().detach() for g in sim]
    input_graphs = [g.cpu().detach() for g in sim[:3]]

    # Calculate box compression factor
    b0 = input_graphs[-2].box_tensor[0]
    b1 = input_graphs[-1].box_tensor[0]
    box_compression_factor = b1 / b0
    
    # Estimate initial box velocity
    current_box_tensor = input_graphs[-1].box_tensor
    current_box_vel_y = estimate_initial_box_vel_y_accurate(
        input_graphs[-3],
        input_graphs[-2],
        input_graphs[-1],
        stride_dt
    )
    
    # Dynamic Parameters
    r0 = sim[0].edge_attr[:, -2]
    num_particles = sim[0].pos.shape[0]
    W_y = c_coupling * num_particles * (stride_dt**2)
    damping = damping_coeff * num_particles * stride_dt

    gt_box = [g.box_tensor for g in input_graphs]
    pred_box = [g.box_tensor for g in input_graphs]

    # Loop through the sequence (skip index 0 as it is the starting state)
    for i in range(len(input_graphs)-1, len(sim) - 1):

        curr_data = sim[i]
        target_pos = sim[i+1].pos

        # Update box X (affine)
        new_box_tensor = current_box_tensor.clone()
        new_box_tensor[0] = new_box_tensor[0] * box_compression_factor

        # Update box
        new_ly, new_vel_y = update_box_y_thermodynamic(
            positions=target_pos,
            edge_index=curr_data.edge_index,
            edge_attr=curr_data.edge_attr,
            current_box=current_box_tensor,
            r0=r0,
            box_vel_y=current_box_vel_y,
            W_y=W_y,
            damping=damping,
            stride_dt=stride_dt,
            temperature=temperature
        )
        
        new_box_tensor[1] = new_ly
        current_box_vel_y = new_vel_y
        current_box_tensor = new_box_tensor.clone()
        
        # Collect results
        pred_box.append(new_box_tensor)
        gt_box.append(sim[i+1].box_tensor)

    return torch.stack(pred_box, dim=0), torch.stack(gt_box, dim=0)


In [ ]:
num_steps = 50

def objective(trial):
    
    # Piston mass scale factor
    C_coupling = trial.suggest_float("C_coupling", 0.1, 10.0, log=False)
    
    # Damping scale factor
    damping_factor = trial.suggest_float("damping_factor", 0.1, 1.5, log=False)

    # Iterate over your validation dataset
    total_error = 0.0
    temperature = 1e-7
    for sim in data["val"]:
        
        sim_strain = (sim[1].box.x - sim[-1].box.x) / sim[0].box.x
        assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
        dump_period = int(assumed_rollout_length / len(sim)) + 1
        stride_dt = dump_period * 0.01 # dump * lammps_dt
        
        pred_box, gt_box = run_trajectory_rollout(
            sim=sim[:num_steps],
            c_coupling=C_coupling,
            damping_coeff=damping_factor,
            stride_dt=stride_dt,
            temperature=temperature,
        )

        error = torch.nn.functional.huber_loss(gt_box[:, 1], pred_box[:, 1])
        total_error += error.item()
        
    return total_error

# Run Optimization
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=300)

print("Best Parameters:")
print(study.best_params)

In [ ]:
random_sim = random.sample(data['test'], k=1)[0]

sim_strain = (random_sim[1].box.x - random_sim[-1].box.x) / random_sim[0].box.x
assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
dump_period = int(assumed_rollout_length / len(random_sim)) + 1
stride_dt = dump_period * 0.01 # dump * lammps_dt

box_pred, box_gt = run_trajectory_rollout(
    sim=random_sim[:50],
    c_coupling=study.best_params['C_coupling'],
    damping_coeff=study.best_params['damping_factor'],
    stride_dt=stride_dt,
    temperature=0.0
)

wrong_box_sim = deepcopy(random_sim)
for g, box in zip(wrong_box_sim, box_pred):
    g.box_tensor = box

pressure_gt = torch.stack([compute_virial_stress(g, r0=random_sim[0].edge_attr[:, -2]).cpu() for g in random_sim[:50]], dim=0)
pressure_default = torch.stack([compute_virial_stress(g, r0=wrong_box_sim[0].edge_attr[:, -2]).cpu() for g in wrong_box_sim[:50]], dim=0)

# region plotting
params = {
    'font.size': 8,                 
    'axes.labelsize': 8,            
    'axes.titlesize': 8,            
    'xtick.labelsize': 8,           
    'ytick.labelsize': 8,   
    'legend.fontsize': 7,           
    'lines.markersize': 3,          
    'figure.figsize': (3.33, 3.33), 
    'figure.dpi': 300,              
    'font.family': 'serif',         
    'axes.formatter.use_mathtext': True, # Renders 1e-4 cleanly as 10^{-4}
}
plt.rcParams.update(params)

fig, ax = plt.subplots(2, 2, layout="constrained", sharex=True)

plot_data = [
    (box_gt[:, 1], box_pred[:, 1], "$L_y$"),
    (pressure_gt[:, 1], pressure_default[:, 1], "$P_{yy}$"),
    (box_gt[:, 0], box_pred[:, 0], "$L_x$"),
    (pressure_gt[:, 0], pressure_default[:, 0], "$P_{xx}$")
]

for i, a in enumerate(ax.flat):
    gt, pred, title = plot_data[i]
    
    a.set_title(title)
    a.plot(gt, label="GT")
    a.plot(pred, label="Est")
    a.legend(frameon=False, loc='best')
    
    # chr(97 + i) dynamically generates 'a', 'b', 'c', 'd'
    a.text(0.05, 0.95, chr(97 + i), 
        transform=a.transAxes, 
        fontsize=params['font.size'], 
        fontweight='bold', 
        va='top', ha='left'
    )

    if i % 2 != 0: 
        a.ticklabel_format(axis='y', style='sci', scilimits=(-2, 2))
    else:
        a.ticklabel_format(axis='y', useOffset=False)

print(f"True P: {-(box_gt[0][1] - box_gt[-1][1])/(box_gt[0][0] - box_gt[-1][0]):.3f}")
print(f"Barostat P: {-(box_pred[0][1] - box_pred[-1][1])/(box_pred[0][0] - box_pred[-1][0]):.3f}")

plt.show()
#endregion


In [ ]:
real_ps = []
pred_ps = []

for sim in data["test"]:
    
    sim_strain = (sim[1].box.x - sim[-1].box.x) / sim[0].box.x
    assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
    dump_period = int(assumed_rollout_length / len(sim)) + 1
    stride_dt = dump_period * 0.01 # dump * lammps_dt

    box_pred, box_gt = run_trajectory_rollout(
        sim=sim,
        c_coupling=study.best_params['C_coupling'],
        damping_coeff=study.best_params['damping_factor'],
        stride_dt=stride_dt,
        temperature=1e-7,
    )

    gt_p = calc_p_ratio_box_tensor(sim).item()
    real_ps.append(gt_p)
    barostat_p = (-(box_pred[-1][1] - box_pred[0][1]) / (box_pred[-1][0] - box_pred[0][0])).item()
    pred_ps.append(barostat_p)

label_text = f"$R^2$={r2_score(real_ps, pred_ps):.5f}"
plt.plot((min(real_ps), max(real_ps)), (min(real_ps), max(real_ps)), linestyle="--", color='black')
plt.scatter(real_ps, pred_ps, s=3, label=label_text)
plt.xlabel("GT Poisson ratio")
plt.ylabel("Barostat Poisson ratio")
plt.legend()
plt.show()